# Energy-Based Reasoning: Graph Shortest-Path Demo

This notebook demonstrates the core EBRM pipeline on the graph shortest-path task:
1. Generate a small graph problem
2. Build encoder, energy network, and decoder
3. Run latent planning (gradient-based optimization in latent space)
4. Visualize the energy landscape and trajectory evolution

In [ ]:
using Random
using Statistics
using Plots

include(joinpath(@__DIR__, "..", "src", "models", "encoder.jl"))
include(joinpath(@__DIR__, "..", "src", "models", "latent_trajectory.jl"))
include(joinpath(@__DIR__, "..", "src", "models", "energy_network.jl"))
include(joinpath(@__DIR__, "..", "src", "models", "decoder.jl"))
include(joinpath(@__DIR__, "..", "src", "inference", "planner.jl"))
include(joinpath(@__DIR__, "..", "data", "graph_reasoning.jl"))
include(joinpath(@__DIR__, "..", "analysis", "visualize.jl"))

using .Encoder, .LatentTrajectory, .EnergyNetwork, .Decoder, .Planner
using .GraphReasoning, .Visualize

## 1. Generate a Graph Problem

In [ ]:
dataset = generate_graph_dataset(; n=50, n_nodes_range=(6, 10), seed=42)
prob = dataset[1]

println("Graph: $(prob.graph.n) nodes, $(count(>(0), prob.weights)) edges")
println("Source: $(prob.src), Destination: $(prob.dst)")
println("Shortest path: $(prob.shortest_path)")
println("Shortest distance: $(round(prob.shortest_dist; digits=3))")

## 2. Build the EBRM System

In [ ]:
latent_dim = 32
T = 6
max_nodes = 10

encoder = GraphEncoder(max_nodes, latent_dim; hidden_dim=64)
energy_model = build_energy_model(latent_dim; hidden_dim=64, n_layers=2)
path_decoder = PathDecoder(latent_dim, max_nodes; max_path_len=max_nodes, hidden_dim=64)

println("System built: d=$latent_dim, T=$T")

## 3. Encode the Problem and Run Latent Planning

In [ ]:
tensors = problem_to_tensors(prob; max_n=max_nodes)
features = vcat(vec(tensors.node_features), vec(tensors.adjacency), tensors.src_onehot, tensors.dst_onehot)

# Use ProblemEncoder since GraphEncoder expects structured input
simple_enc = ProblemEncoder(length(features), 64, latent_dim)
h_x = encode(simple_enc, Float32.(features))

println("Context embedding h_x: $(length(h_x)) dims")
println("h_x range: [$(round(minimum(h_x); digits=3)), $(round(maximum(h_x); digits=3))]")

In [ ]:
rng = MersenneTwister(42)
z_init = 0.01f0 .* randn(rng, Float32, latent_dim, T)
z_init[:, 1] .= h_x

config = PlannerConfig(; steps=100, lr=0.01, use_langevin=true, langevin_noise=0.005)
z_opt, energy_trace = optimize_latent(energy_model, h_x, z_init; config)

println("Planning: $(length(energy_trace)) steps")
println("Energy: $(round(energy_trace[1]; digits=4)) → $(round(energy_trace[end]; digits=4))")

## 4. Visualize Results

In [ ]:
plot_energy_vs_steps(energy_trace; title="Energy During Graph Planning")

In [ ]:
plot_trajectory_2d(z_opt; title="Latent Trajectory (first 2 dims)")

In [ ]:
z_T = z_opt[:, end]
plot_energy_landscape(energy_model, h_x, z_T;
    dims=(1, 2), range_size=1.0, n_grid=40,
    title="Energy Landscape Around z_T")

## 5. Decode the Answer

With an untrained model the decoded path won't be meaningful yet,
but this shows the full pipeline from problem → latent planning → answer.

In [ ]:
logits = decode(path_decoder, z_T)
predicted_path = [argmax(logits[:, t]) for t in 1:length(prob.shortest_path)]

println("True shortest path:  $(prob.shortest_path)")
println("Predicted path:      $(predicted_path)")
println("(Untrained model — accuracy improves with training)")